# Inspección y limpieza — Vacantes de ingeniería de software (LinkedIn)
Dataset: *LinkedIn Software Engineering Jobs* (Kaggle), guardado en `data/raw/postings.csv`.

Objetivo dentro del proyecto de residencias profesionales:
- Analizar **qué tecnologías y competencias demanda el mercado**.
- Servir de base para **recomendar proyectos a los alumnos** según sus habilidades.

La columna clave es `job_skills`: la lista de habilidades que pide cada vacante.

In [1]:
#importar las librerias
import pandas as pd
import numpy as np

In [2]:
#Definir la ruta del archivo
ruta_raw = "../../data/raw/postings.csv"

In [3]:
dataFrame = pd.read_csv(ruta_raw)
dataFrame.head(3)

,job_title,company,job_location,job_link,first_seen,search_city,search_country,job level,job_type,job_summary,job_skills
0,C# Software Engineer,E Tech Group,"West Chester, OH",https://www.linkedin.com/jobs/view/c%23-softwa...,2023-12-25,Covington,United States,Associate,Remote,"At E Tech Group, joining our team means joinin...","C#, .NET, WPF, ASP.NET MVC, WebAPI, C++, Progr..."
1,Software Implementation Engineer,Kardex,"Cincinnati, OH",https://www.linkedin.com/jobs/view/software-im...,2023-12-25,Covington,United States,Associate,Remote,Kardex Remstar is looking for a\nSoftware Impl...,"Software Implementation, Software Testing, Sof..."
2,"Senior Software Engineer, Back End (Go, AWS, J...",Jobs for Humanity,"Chattanooga, TN",https://www.linkedin.com/jobs/view/senior-soft...,2023-12-25,Chattanooga,United States,Mid senior,Onsite,Company Description\nJobs for Humanity is part...,"Java, Python, SQL, Node.js, Go, Scala, AWS, GC..."


## 1. Inspección general

In [4]:
#Dimensiones y tipos de datos
print(dataFrame.shape)
dataFrame.info()

(9380, 11)
<class 'pandas.DataFrame'>
RangeIndex: 9380 entries, 0 to 9379
Data columns (total 11 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   job_title       9380 non-null   str  
 1   company         9380 non-null   str  
 2   job_location    9380 non-null   str  
 3   job_link        9380 non-null   str  
 4   first_seen      9380 non-null   str  
 5   search_city     9380 non-null   str  
 6   search_country  9380 non-null   str  
 7   job level       9380 non-null   str  
 8   job_type        9380 non-null   str  
 9   job_summary     9376 non-null   str  
 10  job_skills      9367 non-null   str  
dtypes: str(11)
memory usage: 806.2 KB


In [5]:
#Valores nulos por columna
dataFrame.isna().sum()

job_title          0
company            0
job_location       0
job_link           0
first_seen         0
search_city        0
search_country     0
job level          0
job_type           0
job_summary        4
job_skills        13
dtype: int64

In [6]:
#Filas duplicadas exactas
dataFrame.duplicated().sum()

np.int64(0)

In [7]:
#Numero de valores distintos por columna
dataFrame.nunique()

job_title         3870
company           3373
job_location      1618
job_link          9380
first_seen           1
search_city        709
search_country       4
job level            2
job_type             3
job_summary       7763
job_skills        9340
dtype: int64

In [8]:
#Valores de las columnas con pocas categorias
for col in ["job level", "job_type", "search_country", "first_seen"]:
    print(col, dataFrame[col].value_counts().to_dict())

job level {'Mid senior': 8101, 'Associate': 1279}
job_type {'Onsite': 4316, 'Hybrid': 2745, 'Remote': 2319}
search_country {'United States': 7335, 'United Kingdom': 1108, 'Canada': 675, 'Australia': 262}
first_seen {'2023-12-25': 9380}


Observaciones:
- `first_seen` tiene **una sola fecha** (2023-12-25): no aporta información y no permite analizar tendencias en el tiempo.
- `search_city` es la ciudad usada para **buscar** en LinkedIn, no la ubicación de la vacante (esa es `job_location`).
- `job_link` es un identificador único de cada vacante.

## 2. Vacantes repetidas
Una misma vacante puede publicarse varias veces (por ejemplo, en varias ciudades). Aunque el enlace cambie, el título, la empresa y la descripción son idénticos.
Si no se eliminan, inflan el conteo de las habilidades de esas empresas.

In [9]:
columnas_vacante = ["job_title", "company", "job_summary"]
repetidas = dataFrame.duplicated(subset=columnas_vacante)
print(f"Vacantes repetidas: {repetidas.sum()} ({repetidas.mean():.1%})")

#Empresas con mas publicaciones repetidas
dataFrame.loc[repetidas, "company"].value_counts().head(5)

Vacantes repetidas: 1536 (16.4%)


company
Canonical                  262
Recruiting from Scratch    188
Affirm                     128
NISC                        61
Grammarly                   60
Name: count, dtype: int64

Además, algunas empresas (sobre todo agregadores como *Jobs for Humanity*) publican **el mismo puesto en decenas de ciudades** con una descripción ligeramente distinta, así que no se detectan como repetidas.
Para medir la demanda real, cada **perfil de puesto de una empresa** (mismo título y misma empresa) se cuenta una sola vez.

In [10]:
columnas_perfil = ["job_title", "company"]
repetidas_perfil = dataFrame.duplicated(subset=columnas_perfil)
print(f"Publicaciones del mismo puesto (título + empresa): {repetidas_perfil.sum()} ({repetidas_perfil.mean():.1%})")
dataFrame.loc[repetidas_perfil, "company"].value_counts().head(5)

Publicaciones del mismo puesto (título + empresa): 2863 (30.5%)


company
Jobs for Humanity          595
Canonical                  265
Recruiting from Scratch    188
Affirm                     130
NISC                        61
Name: count, dtype: int64

## 3. Inspección de las habilidades (`job_skills`)

In [11]:
#Separar la lista de habilidades de cada vacante
habilidades_raw = dataFrame["job_skills"].dropna().str.split(",").explode().str.strip()
habilidades_raw = habilidades_raw[habilidades_raw != ""]

print(f"Menciones de habilidades: {len(habilidades_raw):,}")
print(f"Habilidades distintas:    {habilidades_raw.nunique():,}")
print(f"Aparecen una sola vez:    {(habilidades_raw.value_counts() == 1).sum():,}")

Menciones de habilidades: 181,835
Habilidades distintas:    32,431
Aparecen una sola vez:    22,212


In [12]:
#Ejemplos de una misma tecnologia escrita de formas distintas
for tecnologia in ["react", "node", "javascript", "aws"]:
    variantes = habilidades_raw[habilidades_raw.str.lower().str.contains(tecnologia, regex=False)]
    print(tecnologia, variantes.value_counts().head(6).to_dict())

react {'React': 1529, 'ReactJS': 164, 'React.js': 127, 'React Native': 115, 'React JS': 26, 'React Testing Library': 15}
node {'Node.js': 815, 'NodeJS': 201, 'Node': 187, 'Node.JS': 41, 'Node JS': 32, 'Nodejs': 9}
javascript {'JavaScript': 2405, 'Javascript': 283, 'JavaScript/TypeScript': 26, 'JavaScript frameworks': 17, 'JavaScript Frameworks': 14, 'Javascript/Typescript': 12}
aws {'AWS': 2569, 'AWS Cloud': 68, 'AWS Lambda': 48, 'Amazon Web Services (AWS)': 33, 'AWS services': 28, 'AWS Serverless': 21}


Hay muchas variantes de la misma habilidad (`React`, `ReactJS`, `React.js`, `React JS`…). Hay que **normalizarlas** antes de contar.

## 4. Limpieza de las columnas generales

In [13]:
#Trabajar sobre una copia para conservar los datos originales
df_limpio = dataFrame.copy()

#Quitar espacios sobrantes en las columnas de texto
for col in df_limpio.select_dtypes(exclude="number").columns:
    df_limpio[col] = df_limpio[col].str.strip()

#Eliminar las vacantes sin habilidades: no sirven para el analisis
df_limpio = df_limpio.dropna(subset=["job_skills"])

#Eliminar las vacantes repetidas y los puestos publicados varias veces (se conserva la primera publicacion)
df_limpio = df_limpio.drop_duplicates(subset=columnas_vacante, keep="first")
df_limpio = df_limpio.drop_duplicates(subset=columnas_perfil, keep="first")
print(df_limpio.shape)

(6507, 11)


In [14]:
#Eliminar columnas que no aportan al analisis
#  - job_link: identificador; se reemplaza por un id numerico
#  - first_seen: tiene una sola fecha
#  - search_city: es un parametro de busqueda, no la ubicacion de la vacante
#  - job_summary: texto largo en ingles; ya se uso para detectar repetidas (sigue disponible en data/raw)
df_limpio = df_limpio.drop(columns=["job_link", "first_seen", "search_city", "job_summary"])
df_limpio = df_limpio.reset_index(drop=True)
df_limpio.insert(0, "id_vacante", df_limpio.index + 1)
df_limpio.head(3)

,id_vacante,job_title,company,job_location,search_country,job level,job_type,job_skills
0,1,C# Software Engineer,E Tech Group,"West Chester, OH",United States,Associate,Remote,"C#, .NET, WPF, ASP.NET MVC, WebAPI, C++, Progr..."
1,2,Software Implementation Engineer,Kardex,"Cincinnati, OH",United States,Associate,Remote,"Software Implementation, Software Testing, Sof..."
2,3,"Senior Software Engineer, Back End (Go, AWS, J...",Jobs for Humanity,"Chattanooga, TN",United States,Mid senior,Onsite,"Java, Python, SQL, Node.js, Go, Scala, AWS, GC..."


## 5. Normalización de las habilidades
Proceso en tres pasos:
1. **Clave normalizada**: minúsculas, sin espacios, puntos ni guiones, y sin el sufijo `js` (`React.js` → `react`, `Node JS` → `node`).
2. **Sinónimos**: se unifican nombres distintos de la misma tecnología (`Golang` → `Go`, `Microsoft Azure` → `Azure`).
3. **Nombre a mostrar**: para cada clave se usa la escritura más frecuente en los datos (`JavaScript` en lugar de `Javascript`).

In [15]:
def clave_habilidad(texto):
    clave = texto.lower()
    for caracter in [" ", ".", "-", "_"]:
        clave = clave.replace(caracter, "")
    #Quitar el sufijo "js" solo si queda algo antes (React.js -> react, pero "JS" se conserva)
    if clave.endswith("js") and len(clave) > 2:
        clave = clave[:-2]
    return clave

#Sinonimos detectados en los datos: clave original -> clave unificada
sinonimos = {
    "js": "javascript",
    "awscloud": "aws",
    "amazonwebservices": "aws",
    "amazonwebservices(aws)": "aws",
    "googlecloud": "gcp",
    "googlecloudplatform": "gcp",
    "googlecloudplatform(gcp)": "gcp",
    "microsoftazure": "azure",
    "golang": "go",
    "k8s": "kubernetes",
    "postgres": "postgresql",
    "machinelearning(ml)": "machinelearning",
    "ml": "machinelearning",
    "ai": "artificialintelligence",
    "artificialintelligence(ai)": "artificialintelligence",
    "rest": "restapi",
    "restapis": "restapi",
    "restfulapi": "restapi",
    "restfulapis": "restapi",
    "cicd": "ci/cd",
    "cicdpipelines": "ci/cd",
    "ci/cdpipelines": "ci/cd",
    "llm": "llms",
    "largelanguagemodels": "llms",
    "largelanguagemodels(llms)": "llms",
    "mssql": "sqlserver",
    "oop": "objectorientedprogramming",
    "agilemethodologies": "agile",
    "agilemethodology": "agile",
    "agiledevelopment": "agile",
    "agilepractices": "agile",
    "agilesoftwaredevelopment": "agile",
    "communicationskills": "communication",
    "tdd": "testdrivendevelopment",
    "sdlc": "softwaredevelopmentlifecycle",
    "codereviews": "codereview",
    "apis": "api",
    "nosqldatabases": "nosql",
    "html5": "html",
    "css3": "css",
    "springframework": "spring",
    "cloud": "cloudcomputing",
    "cloudservices": "cloudcomputing",
    "automatedtesting": "testautomation",
    "rdbms": "relationaldatabases",
    "sourcecontrol": "versioncontrol",
}

#Nombres a mostrar fijados a mano cuando la escritura mas frecuente no es la mas clara
nombres_fijos = {
    "restapi": "REST API",
    "gcp": "GCP",
    "artificialintelligence": "Artificial Intelligence",
    "llms": "LLMs",
    "agile": "Agile",
    "api": "APIs",
    "cloudcomputing": "Cloud Computing",
    "testautomation": "Test Automation",
    "objectorientedprogramming": "Object-Oriented Programming",
    "objectorienteddesign": "Object-Oriented Design",
    "codereview": "Code Review",
    "testdrivendevelopment": "Test Driven Development",
    "softwaredevelopmentlifecycle": "Software Development Life Cycle",
}

def unificar(texto):
    clave = clave_habilidad(texto)
    return sinonimos.get(clave, clave)

In [16]:
#Una fila por cada habilidad de cada vacante (formato largo)
vacante_habilidad = (
    df_limpio[["id_vacante", "job_skills"]]
    .assign(original=df_limpio["job_skills"].str.split(","))
    .explode("original")
)
vacante_habilidad["original"] = vacante_habilidad["original"].str.strip()
vacante_habilidad = vacante_habilidad[vacante_habilidad["original"] != ""]
vacante_habilidad["clave"] = vacante_habilidad["original"].map(unificar)

#Nombre a mostrar: la escritura mas frecuente de cada clave (o el nombre fijado a mano)
nombre_mostrar = (
    vacante_habilidad.groupby("clave")["original"].agg(lambda s: s.value_counts().index[0])
)
nombre_mostrar.update(pd.Series(nombres_fijos))
vacante_habilidad["habilidad"] = vacante_habilidad["clave"].map(nombre_mostrar)

#Una vacante puede repetir la misma habilidad con dos escrituras: se cuenta una vez
vacante_habilidad = vacante_habilidad.drop_duplicates(subset=["id_vacante", "clave"])

print(f"Habilidades distintas antes: {habilidades_raw.nunique():,}")
print(f"Habilidades distintas despues de normalizar: {vacante_habilidad['clave'].nunique():,}")

Habilidades distintas antes: 32,431
Habilidades distintas despues de normalizar: 23,888


In [17]:
#Comprobar la normalizacion con los ejemplos de antes
ejemplos = vacante_habilidad[vacante_habilidad["clave"].isin(["react", "node", "javascript", "aws", "go", "restapi"])]
ejemplos.groupby("habilidad")["original"].agg(lambda s: s.value_counts().head(5).to_dict())

habilidad
AWS           {'AWS': 1513, 'Amazon Web Services': 40, 'AWS ...
Go            {'Go': 349, 'Golang': 152, 'GoLang': 22, 'GO': 7}
JavaScript    {'JavaScript': 1548, 'Javascript': 210, 'Java ...
Node.js       {'Node.js': 508, 'NodeJS': 159, 'Node': 72, 'N...
REST API      {'REST': 318, 'RESTful APIs': 219, 'REST APIs'...
React         {'React': 1087, 'ReactJS': 105, 'React.js': 10...
Name: original, dtype: object

## 6. Filtrar habilidades poco frecuentes
Muchas "habilidades" aparecen en muy pocas vacantes: son frases sueltas o errores de captura y añaden ruido.
Se conservan las que aparecen en al menos `MIN_VACANTES` vacantes.

In [18]:
#Numero de vacantes en las que aparece cada habilidad
catalogo = (
    vacante_habilidad.groupby(["clave", "habilidad"]).size()
    .rename("num_vacantes").reset_index()
    .sort_values("num_vacantes", ascending=False)
)

#Cobertura segun el umbral elegido
for umbral in [5, 10, 20, 50]:
    frecuentes = catalogo[catalogo["num_vacantes"] >= umbral]
    cobertura = frecuentes["num_vacantes"].sum() / catalogo["num_vacantes"].sum()
    print(f"Umbral {umbral:>2}: {len(frecuentes):>5} habilidades, cubren el {cobertura:.1%} de las menciones")

Umbral  5:  2470 habilidades, cubren el 77.3% de las menciones
Umbral 10:  1292 habilidades, cubren el 71.3% de las menciones
Umbral 20:   692 habilidades, cubren el 64.9% de las menciones
Umbral 50:   304 habilidades, cubren el 55.5% de las menciones


In [19]:
#Umbral elegido: buen equilibrio entre conservar informacion y quitar ruido
MIN_VACANTES = 10

catalogo["incluida"] = catalogo["num_vacantes"] >= MIN_VACANTES
vacante_habilidad = vacante_habilidad[vacante_habilidad["clave"].isin(catalogo.loc[catalogo["incluida"], "clave"])]

print(f"Habilidades incluidas: {catalogo['incluida'].sum():,}")
catalogo.head(15)

Habilidades incluidas: 1,292


,clave,habilidad,num_vacantes,incluida
16723,python,Python,2169,True
562,agile,Agile,2073,True
11128,java,Java,1914,True
11200,javascript,JavaScript,1772,True
2071,aws,AWS,1609,True
3125,c#,C#,1410,True
20249,sql,SQL,1373,True
17038,react,React,1328,True
9046,git,Git,1308,True
3148,c++,C++,1221,True


In [20]:
#Reconstruir la lista de habilidades limpia de cada vacante
habilidades_por_vacante = vacante_habilidad.groupby("id_vacante")["habilidad"].agg("; ".join)
df_limpio["habilidades"] = df_limpio["id_vacante"].map(habilidades_por_vacante)
df_limpio["num_habilidades"] = df_limpio["id_vacante"].map(vacante_habilidad.groupby("id_vacante").size()).fillna(0).astype(int)
df_limpio = df_limpio.drop(columns=["job_skills"])

#Vacantes que se quedaron sin ninguna habilidad frecuente
print(f"Vacantes sin habilidades tras el filtro: {(df_limpio['num_habilidades'] == 0).sum()}")
df_limpio = df_limpio[df_limpio["num_habilidades"] > 0].reset_index(drop=True)
df_limpio["num_habilidades"].describe().round(1)

Vacantes sin habilidades tras el filtro: 18


count    6489.0
mean       13.9
std         6.9
min         1.0
25%         9.0
50%        13.0
75%        18.0
max        64.0
Name: num_habilidades, dtype: float64

## 7. Clasificación de las habilidades por categoría
Se clasifican a mano las habilidades más frecuentes. Esto permite separar las **tecnologías** de las **competencias blandas** y de los **conceptos generales** (como *Software Engineering*), que no son una tecnología concreta.
Las habilidades no clasificadas quedan como `Sin clasificar`.

In [21]:
categorias = {
    "Lenguajes": [
        "Python", "Java", "JavaScript", "SQL", "C#", "C++", "TypeScript", "Go", "HTML", "CSS", "C",
        "C/C++", "Scala", "PHP", "Kotlin", "Bash", "Rust", "Ruby", "MATLAB", "PowerShell", "Perl",
        "Swift", "TSQL", "HTML/CSS",
    ],
    "Frameworks y librerías": [
        "React", "Node.js", "Angular", ".NET", "Spring Boot", ".NET Core", "Vue.js", "ASP.NET",
        "jQuery", "Spring", "Django", "Entity Framework", "Bootstrap", "Redux", "Hibernate", "Next.js",
        "WPF", ".NET Framework", "JUnit", "Jest", "Selenium",
    ],
    "Cloud": ["AWS", "Azure", "GCP", "Cloud Computing", "S3", "Lambda", "EC2", "SaaS"],
    "DevOps e infraestructura": [
        "Git", "Docker", "Kubernetes", "Linux", "CI/CD", "DevOps", "Jenkins", "Terraform", "GitHub",
        "Azure DevOps", "GitLab", "Ansible", "Unix", "Windows", "Bitbucket", "Maven", "Version Control",
        "Containerization", "SVN", "DevSecOps", "Continuous Integration", "Continuous Delivery",
        "Visual Studio", "Networking", "TCP/IP", "Operating Systems",
    ],
    "Bases de datos": [
        "PostgreSQL", "SQL Server", "NoSQL", "MySQL", "MongoDB", "Oracle", "Redis", "Elasticsearch",
        "Relational databases", "Databases", "DynamoDB", "Cassandra", "Data Modeling",
    ],
    "Datos e IA": [
        "Machine Learning", "Artificial Intelligence", "Data Analysis", "Data Science", "Data Analytics",
        "Big Data", "Spark", "Kafka", "RabbitMQ", "LLMs",
    ],
    "Arquitectura y buenas prácticas": [
        "REST API", "Microservices", "Unit Testing", "Object-Oriented Programming", "Software Design",
        "Software Testing", "Software Architecture", "Design Patterns", "Distributed Systems",
        "Integration testing", "Test Driven Development", "GraphQL", "Web Services", "Object-Oriented Design",
        "MVC", "APIs", "Test Automation", "SOAP", "Code Review", "System Design", "Data structures",
        "Algorithms", "Debugging", "Scalability", "Security", "Cybersecurity", "JSON", "XML", "Web API",
        "API Development", "Quality Assurance", "BDD", "UML", "AJAX", "Network security",
    ],
    "Metodologías y gestión": [
        "Agile", "Scrum", "Jira", "Confluence", "Kanban", "Software Development Life Cycle", "Project Management",
    ],
    "Sistemas embebidos": [
        "Embedded Systems", "RTOS", "SPI", "I2C", "CAN", "Embedded Software", "Embedded Software Development",
        "Robotics",
    ],
    "Desarrollo móvil": ["Android", "iOS", "React Native"],
    "Competencias blandas": [
        "Communication", "Problem Solving", "Teamwork", "Collaboration", "Leadership", "Mentoring",
        "Analytical Skills", "Documentation", "Troubleshooting", "People Management",
    ],
    "Conceptos generales": [
        "Software Engineering", "Software Development", "Computer Science", "Computer Engineering",
        "Electrical Engineering", "Programming", "Coding", "Engineering", "Design", "Web Development",
        "Full Stack Development", "Frontend development", "Backend development", "Programming Languages",
        "Open source frameworks", "Testing", "Automation", "Open source", "Software Maintenance",
    ],
}

categoria_de = {habilidad: categoria for categoria, lista in categorias.items() for habilidad in lista}
catalogo["categoria"] = catalogo["habilidad"].map(categoria_de).fillna("Sin clasificar")
vacante_habilidad["categoria"] = vacante_habilidad["habilidad"].map(categoria_de).fillna("Sin clasificar")

#Comprobar que todos los nombres de la clasificacion existen entre las habilidades incluidas
no_encontradas = set(categoria_de) - set(catalogo.loc[catalogo["incluida"], "habilidad"])
print("Nombres no encontrados:", no_encontradas or "ninguno")

Nombres no encontrados: ninguno


In [22]:
#Cuantas menciones quedan clasificadas
menciones = vacante_habilidad["categoria"].value_counts()
print(f"Menciones clasificadas: {1 - menciones.get('Sin clasificar', 0) / menciones.sum():.1%}")
menciones

Menciones clasificadas: 69.3%


categoria
Sin clasificar                     27646
Lenguajes                          15421
DevOps e infraestructura            9427
Arquitectura y buenas prácticas     8945
Frameworks y librerías              6428
Conceptos generales                 4923
Metodologías y gestión              4280
Cloud                               3942
Bases de datos                      3546
Competencias blandas                2619
Datos e IA                          1723
Sistemas embebidos                   851
Desarrollo móvil                     313
Name: count, dtype: int64

## 8. Traducción al español

In [23]:
#Traducir los valores categoricos
traduccion_valores = {
    "job level": {"Associate": "Asociado", "Mid senior": "Intermedio-Senior"},
    "job_type": {"Onsite": "Presencial", "Hybrid": "Híbrido", "Remote": "Remoto"},
    "search_country": {"United States": "Estados Unidos", "United Kingdom": "Reino Unido",
                       "Canada": "Canadá", "Australia": "Australia"},
}
for col, mapeo in traduccion_valores.items():
    df_limpio[col] = df_limpio[col].map(mapeo)

#Comprobar que ningun valor quedo sin traducir
df_limpio[list(traduccion_valores)].isna().sum().sum()

np.int64(0)

In [24]:
#Traducir los nombres de las columnas
traduccion_columnas = {
    "job_title": "titulo",
    "company": "empresa",
    "job_location": "ubicacion",
    "search_country": "pais",
    "job level": "nivel",
    "job_type": "modalidad",
}
df_limpio = df_limpio.rename(columns=traduccion_columnas)
df_limpio.columns.tolist()

['id_vacante',
 'titulo',
 'empresa',
 'ubicacion',
 'pais',
 'nivel',
 'modalidad',
 'habilidades',
 'num_habilidades']

## 9. Puestos de nivel inicial
Los residentes buscan puestos de entrada. Se marca como **puesto inicial** toda vacante de nivel `Asociado` o cuyo título indique prácticas o recién egresados.
Los nombres de las habilidades y los títulos se mantienen en inglés: son términos técnicos universales.

In [25]:
palabras_inicial = r"\b(?:intern|internship|junior|jr|entry|graduate|trainee|apprentice)\b"
titulo_inicial = df_limpio["titulo"].str.contains(palabras_inicial, case=False, regex=True)
df_limpio["es_puesto_inicial"] = (df_limpio["nivel"] == "Asociado") | titulo_inicial

df_limpio["es_puesto_inicial"].value_counts()

es_puesto_inicial
False    5388
True     1101
Name: count, dtype: int64

In [26]:
#Revision final
df_limpio.info()
df_limpio.head()

<class 'pandas.DataFrame'>
RangeIndex: 6489 entries, 0 to 6488
Data columns (total 10 columns):
 #   Column             Non-Null Count  Dtype
---  ------             --------------  -----
 0   id_vacante         6489 non-null   int64
 1   titulo             6489 non-null   str  
 2   empresa            6489 non-null   str  
 3   ubicacion          6489 non-null   str  
 4   pais               6489 non-null   str  
 5   nivel              6489 non-null   str  
 6   modalidad          6489 non-null   str  
 7   habilidades        6489 non-null   str  
 8   num_habilidades    6489 non-null   int64
 9   es_puesto_inicial  6489 non-null   bool 
dtypes: bool(1), int64(2), str(7)
memory usage: 462.7 KB


,id_vacante,titulo,empresa,ubicacion,pais,nivel,modalidad,habilidades,num_habilidades,es_puesto_inicial
0,1,C# Software Engineer,E Tech Group,"West Chester, OH",Estados Unidos,Asociado,Remoto,C#; .NET; WPF; ASP.NET MVC; Web API; C++; SQL ...,13,True
1,2,Software Implementation Engineer,Kardex,"Cincinnati, OH",Estados Unidos,Asociado,Remoto,Software Implementation; Software Testing; Sof...,7,True
2,3,"Senior Software Engineer, Back End (Go, AWS, J...",Jobs for Humanity,"Chattanooga, TN",Estados Unidos,Intermedio-Senior,Presencial,Java; Python; SQL; Node.js; Go; Scala; AWS; GC...,12,False
3,4,"Senior Manager, Software Engineering, Full Stack",Jobs for Humanity,"Chattanooga, TN",Estados Unidos,Intermedio-Senior,Presencial,JavaScript; Java; TypeScript; SQL; Python; Go;...,11,False
4,5,"Lead Software Engineer, Full Stack(JavaScript/...",Jobs for Humanity,"Chattanooga, TN",Estados Unidos,Intermedio-Senior,Presencial,JavaScript; Java; AWS; Vue.js; HTML/CSS; TypeS...,14,False


## 10. Guardar datos limpios
Se guardan tres archivos:
- `vacantes_limpio.csv`: una fila por vacante, con su lista de habilidades limpia.
- `vacantes_habilidades.csv`: una fila por cada par vacante–habilidad (cómodo para contar y para el recomendador).
- `catalogo_habilidades.csv`: todas las habilidades normalizadas con su frecuencia y si se incluyeron.
- `normalizacion_habilidades.json`: sinónimos, nombres y categorías, para normalizar habilidades nuevas (lo usa el backend).

In [27]:
import os

#Definir la carpeta de salida
carpeta_processed = "../../data/processed"
os.makedirs(carpeta_processed, exist_ok=True)

df_limpio.to_csv(os.path.join(carpeta_processed, "vacantes_limpio.csv"), index=False)
vacante_habilidad[["id_vacante", "habilidad", "categoria"]].to_csv(
    os.path.join(carpeta_processed, "vacantes_habilidades.csv"), index=False)
catalogo[["habilidad", "categoria", "num_vacantes", "incluida"]].to_csv(
    os.path.join(carpeta_processed, "catalogo_habilidades.csv"), index=False)

#Diccionario de normalizacion: lo usa el backend para entender habilidades escritas de cualquier forma
import json
normalizacion = {
    "sinonimos": sinonimos,
    "nombres": catalogo.loc[catalogo["incluida"]].set_index("clave")["habilidad"].to_dict(),
    "categorias": catalogo.loc[catalogo["incluida"]].set_index("habilidad")["categoria"].to_dict(),
}
with open(os.path.join(carpeta_processed, "normalizacion_habilidades.json"), "w", encoding="utf-8") as f:
    json.dump(normalizacion, f, ensure_ascii=False, indent=1)

print(f"vacantes_limpio.csv:       {df_limpio.shape}")
print(f"vacantes_habilidades.csv:  {len(vacante_habilidad):,} filas")
print(f"catalogo_habilidades.csv:  {len(catalogo):,} habilidades ({catalogo['incluida'].sum():,} incluidas)")

vacantes_limpio.csv:       (6489, 10)
vacantes_habilidades.csv:  90,064 filas
catalogo_habilidades.csv:  23,888 habilidades (1,292 incluidas)
